### Feature Engineering (Проектирование/конструирование признаков) — это процесс преобразования необработанных данных в признаки, подходящие для моделей машинного обучения. Другими словами, это процесс выбора, извлечения и преобразования наиболее релевантных признаков из доступных данных для построения более точных и эффективных моделей машинного обучения.

Успех моделей машинного обучения во многом зависит от качества признаков, используемых для их обучения. Инженерия признаков включает в себя набор методов, которые позволяют нам создавать новые признаки путем комбинирования или преобразования существующих. Эти методы помогают выделить наиболее важные закономерности и взаимосвязи в данных, что, в свою очередь, помогает модели машинного обучения более эффективно обучаться на основе данных.

Задачи Feature Engineering:
Преобразование сырых данных в информативные признаки.
Добавление контекстных новых признаков.
Кодирование категориальных признаков (Encoding).
Работа с числовыми признаками (масштабирование, логарифмирование, создание полиномиальных признаков и пр.).
Создание агрегированных признаков.
Работа с пропусками и аномалиями и пр.
Feature Engineering — это как конструктор, где мы строим полезные признаки, чтобы дать модели больше информации.

![alt text](asset-v1_SkillFactory+MIFIML-2sem+2025+type@asset+block@Классическое_МО_M6_img1.png)

Например, в прошлом модуле вы рассматривали Kernel trick для алгоритма SVM. Это использование ядерной функции, которая берет низкоразмерное входное пространство и преобразует его в более высокоразмерное, то есть преобразует неразрешимую проблему в разрешимую. Это задача также относится к Feature Engineering.

Рассмотрим задачу Feature Engineering на прикладном примере, чтобы понять до конца, что это и как работает. Хотя со многими элементами вы уже знакомились в предыдущих модулях, в этом постараемся систематизировать все знания и посмотреть более сложные примеры, а также обсудим создание пайплайнов для решения задач ML.

Загрузим датасет с Kaggle. Данный набор данных содержит информацию о потреблении электричества на одной из подстанций Москвы до 2008 года включительно. А в рамках соревнования требуется построить предсказание потребления электричества на первую половину 2009 года.

In [8]:
'''import opendatasets as od'''
import pandas as pd

'''od.download("https://www.kaggle.com/competitions/electricity-consumption")'''
df = pd.read_csv(r"C:\IDE\2 семестр\Классическое МО\data\electricity-consumption\train.csv")

In [ ]:
            datetime	      total
0	01.01.2005 00:00:00	NaN
1	01.01.2005 01:00:00	154139.8084
2	01.01.2005 02:00:00	157818.3593
3	01.01.2005 03:00:00	149310.6991
4	01.01.2005 04:00:00	138282.0380

Видим, что есть всего два столбца: временная метка (datetime) и целевая переменная — потребление электроэнергии (total). Причем уже на первой строке виден пропуск в данных, что может стать проблемой при обучении модели.

Проблемы с данными:
Нехватка признаков.
У нас нет никаких дополнительных признаков, которые могли бы помочь модели понять закономерности в потреблении электроэнергии.

Например, день недели, месяц, сезон или информация о том, выходной ли это день — все это может существенно повлиять на предсказания.
Пропуски в данных.
Как видно из примера, уже на первой строке есть пропуск, и, возможно, они встречаются и дальше. Если оставить это без внимания, модель может обучиться некорректно или вовсе не запуститься.

Сырые временные ряды.
Временной формат сам по себе неинформативен. Модели не понимают, что за числа в датах — для них это просто текст или числа без явного смысла. Нам нужно извлечь из времени полезные признаки.

Почему плохие или малоинформативные данные — это проблема?

Когда есть мало переменных или они слабо связаны с целевой переменной, модель может оказаться неспособной найти нужные зависимости. Она либо переобучится на шум, либо будет давать случайные предсказания. Даже мощные алгоритмы, такие как градиентный бустинг, не смогут показать хороший результат без качественного набора признаков.

В рамках лекций Feature Engineering разберемся, как извлечь максимум информации из доступных данных.

Теперь посмотрим более подробную информацию о датасете:

In [ ]:
df.info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35064 entries, 0 to 35063
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   datetime  35064 non-null  object 
 1   total     34946 non-null  float64
dtypes: float64(1), object(1)
memory usage: 548.0+ KB

In [9]:
'''Уже обсудили, что в датасете есть пропуски — их просто удалим. Также видим, что колонка datetime дана в виде строки:
'''
# просто удаляем пропуски
df = df[~df['total'].isna()]

In [10]:
'''Когда вы работаете с временными рядами, одно из первых действий — приведение даты и времени к правильному формату. В целом, приведение данных к верному типу данных справедливо для любого датасета: если визуально в датасете число, это не значит, что значение действительно соответствует числовому типу данных.
'''
df['datetime'] = pd.to_datetime(df['datetime'], format="%d.%m.%Y %H:%M:%S")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34946 entries, 1 to 35063
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  34946 non-null  datetime64[ns]
 1   total     34946 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 819.0 KB


Преобразование даты.
Мы используем pd.to_datetime, чтобы преобразовать столбец datetime из текстового формата в формат datetime. Это позволяет работать с датами как с числовыми значениями, извлекать временные признаки (год, месяц, день, час) и производить различные временные операции.

Формат даты.
Указываем формат даты явно — %d.%m.%Y %H:%M:%S,
где:

%d — день,
%m — месяц,
%Y — год,
%H — час,
%M — минута,
%S — секунда.
Это важно, чтобы избежать ошибок при неправильном считывании формата.
<class 'pandas.core.frame.DataFrame'>
Index: 34946 entries, 1 to 35063
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  34946 non-null  datetime64[ns]
 1   total     34946 non-null  int64         
dtypes: datetime64[ns](1), int64(1)
memory usage: 819.0 KB
Теперь перейдем непосредственно к Feature Engineering и научимся извлекать полезную информацию из имеющихся данных. В данном случае достанем все временные признаки (Time-based Features):

hour — час (можно выявить пиковые часы нагрузки).
day — день месяца.
month — месяц (сезонность).
day_of_week — день недели (выходные и будни).
is_weekend — выходной или нет (0 или 1).
quarter — квартал года.
season — сезон (зима, весна, лето, осень).
is_holiday — праздничный день (по календарю праздников).
Зачем это нужно? Во-первых, даже из жизненной логики нетрудно догадаться, что в вечернее время скорее всего потребление электричества выше, как и в выходные дни, как и в зимнее время.

In [11]:
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['season'] = df['datetime'].dt.quarter
df['day'] = df['datetime'].dt.day
df['hour'] = df['datetime'].dt.hour
df['dayofyear'] = df['datetime'].dt.day_of_year
df['dayofweek'] = df['datetime'].dt.day_of_week
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

df.head()

,datetime,total,year,month,season,day,hour,dayofyear,dayofweek,is_weekend
1,2005-01-01 01:00:00,154139.8084,2005,1,1,1,1,1,5,1
2,2005-01-01 02:00:00,157818.3593,2005,1,1,1,2,1,5,1
3,2005-01-01 03:00:00,149310.6991,2005,1,1,1,3,1,5,1
4,2005-01-01 04:00:00,138282.0380,2005,1,1,1,4,1,5,1
5,2005-01-01 05:00:00,132032.7479,2005,1,1,1,5,1,5,1


Какие еще признаки можно добавить и сконструировать?

Добавление тригонометрических функций к данным полезно, когда в данных присутствуют циклические признаки (например, время суток, дни недели, месяцы и пр.). Эти циклические признаки имеют естественную структуру, где значения «заворачиваются» (например, после 23 часов следует 0, а после декабря — январь). Применение тригонометрических функций позволяет модели лучше учитывать эту цикличность.

In [12]:
import math
import numpy as np

#добавление тригонометрических функций от данных
def add_trigonom(df):
    df['sin_month'] = df['month'].apply(math.sin)
    df['cos_month'] = df['month'].apply(math.cos)

    df['sin_hour'] = df['hour'].apply(math.sin)
    df['cos_hour'] = df['hour'].apply(math.cos)

    df['sin**2_hour'] = df['sin_hour'] * df['sin_hour']
    df['cos**2_hour'] = df['cos_hour'] * df['cos_hour']

    df['hour_sin'] = (df['hour'] / 23 * 2 * np.pi).apply(math.sin)
    df['hour_cos'] = (df['hour'] / 23 * 2 * np.pi).apply(math.cos)

    df['month_sin'] = ((df['month'] - 1) / 11 * 2 * np.pi).apply(math.sin)
    df['month_cos'] = ((df['month'] - 1) / 11 * 2 * np.pi).apply(math.cos)

    df['hour_sin**2'] = df['hour_sin'] * df['hour_sin']
    df['month_cos**2'] = df['month_cos'] * df['month_cos']

    day = 24
    year = 365.2425*day

    df['day_sin'] = (df.index.hour * 2 * np.pi / day)
    df['day_sin'] = df['day_sin'].apply(math.sin)
    df['day_cos'] = (df.index.hour * 2 * np.pi / day)
    df['day_cos'] = df['day_cos'].apply(math.cos)
    df['year_sin'] = (df.index.hour * 2 * np.pi / year)
    df['year_sin'] = df['year_sin'].apply(math.sin)
    df['year_cos'] = (df.index.hour * 2 * np.pi / year)
    df['year_cos'] = df['year_cos'].apply(math.cos)

    return df

df = df.set_index('datetime')
df = add_trigonom(df)

df
df = df.reset_index(drop=True)
df = df.astype({'total': 'int32'})

In [13]:
df.to_csv('my_data_to_FS.csv', index=False, sep=',', encoding='utf-8')

sin_month, cos_month — синус и косинус номера месяца.
sin_hour, cos_hour — синус и косинус часа.
Эти признаки помогают отразить периодичность месяца и суток.

Квадраты тригонометрических функций.
sin**2_hour, cos**2_hour — квадраты синуса и косинуса часа.
Аналогично для месяца — hour_sin**2, month_cos**2.
Эти признаки могут улучшить качество модели, помогая ей лучше улавливать сложные зависимости.

Нормализованные тригонометрические признаки.
hour_sin, hour_cos — синус и косинус, нормализованные на диапазон [0, 1] для часов.
month_sin, month_cos — аналогично для месяцев.
Суточные и годовые циклы.
day_sin, day_cos — синус и косинус часа в пределах суток.
year_sin, year_cos — синус и косинус часа в пределах года.
Это особенно полезно, если потребление энергии зависит, например, от времени суток или времени года.

Машинные модели плохо понимают цикличность, а тригонометрические функции отлично ее отображают. Например, sin(0) и sin(2π) оба равны нулю, что отражает начало и конец суток.

В итоге получаем много новых признаков, которые помогут модели лучше понять временные закономерности в данных.